# 1982 — John Hopfield
## Hopfield Network: เครือข่ายที่ทำหน้าที่เป็น "หน่วยความจำ"

| | |
|---|---|
| **ผู้คิดค้น** | John J. Hopfield — นักฟิสิกส์, Caltech |
| **ผลงาน** | *"Neural networks and physical systems with emergent collective computational abilities"* — PNAS, 1982 |
| **แนวคิดหลัก** | **Associative Memory (หน่วยความจำแบบเชื่อมโยง)** — ให้ข้อมูลที่ขาดหาย/มี noise เข้าไป เครือข่ายจะ "นึกออก" และคืนรูปแบบเต็มที่จำไว้ |
| **ความสำคัญ** | นำแนวคิดจากฟิสิกส์ (**พลังงาน**, spin glass) มาอธิบาย Neural Network ทำให้วงการกลับมาคึกคักหลัง AI Winter |
| **รางวัล** | **Nobel Prize in Physics 2024** ร่วมกับ Geoffrey Hinton สำหรับการค้นพบพื้นฐานที่ทำให้ machine learning ด้วย neural network เป็นไปได้ |

> 🧠 **ตัวอย่างในชีวิตจริง:** ได้ยินเพลงแค่ 2–3 โน้ตแรกก็นึกออกทั้งเพลง — สมองไม่ได้ "ค้นหา" แต่ "ไหลลงสู่" ความทรงจำที่ใกล้ที่สุด

## 1. แบบจำลอง

- มีเซลล์ $N$ ตัว แต่ละตัวมีสถานะ $s_i \in \{-1, +1\}$ และ **ทุกตัวเชื่อมกับทุกตัว** (ไม่มีชั้น input/output แยก)
- weight สมมาตร $w_{ij} = w_{ji}$ และไม่เชื่อมกับตัวเอง $w_{ii} = 0$

### จดจำ (Hebbian learning — "cells that fire together, wire together")

$$
w_{ij} = \sum_{\mu=1}^{P} p_i^{\mu}\, p_j^{\mu} \quad (i \ne j)
\qquad\text{หรือเขียนเป็นเมทริกซ์}\qquad
W = \sum_{\mu} \mathbf{p}^{\mu} (\mathbf{p}^{\mu})^{\top} - P\,I
$$

(หลายตำราหารด้วย $N$ ด้วย — ไม่เปลี่ยนเครื่องหมายของผลลัพธ์ จึงไม่เปลี่ยนการทำงาน)

### นึก (Asynchronous update — อัปเดตทีละเซลล์)

$$
h_i = \sum_j w_{ij} s_j, \qquad
s_i \leftarrow \begin{cases} +1 & h_i > 0 \\ -1 & h_i < 0 \\ s_i \text{ (คงเดิม)} & h_i = 0 \end{cases}
$$

### ฟังก์ชันพลังงาน (Energy)

$$
E(\mathbf{s}) = -\frac{1}{2} \sum_{i,j} w_{ij}\, s_i s_j = -\frac{1}{2}\, \mathbf{s}^{\top} W \mathbf{s}
$$

**ทฤษฎีบทสำคัญ:** การอัปเดตทีละเซลล์ **ไม่มีวันทำให้ $E$ เพิ่มขึ้น** → ระบบไหลลง "หุบเขา" พลังงานจนหยุดนิ่ง และรูปแบบที่จำไว้คือก้นหุบเขา

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def hebbian_weights(patterns, normalize=False):
    P = np.array(patterns)
    W = P.T @ P
    np.fill_diagonal(W, 0)
    return W / P.shape[1] if normalize else W


def energy(W, s):
    return -0.5 * s @ W @ s


def recall(W, s, rng=None, max_sweeps=20, verbose=False):
    s = s.copy()
    energies = [float(energy(W, s))]
    for sweep in range(max_sweeps):
        changed = False
        order = rng.permutation(len(s)) if rng is not None else range(len(s))
        for i in order:
            h = W[i] @ s
            new = s[i] if h == 0 else (1 if h > 0 else -1)
            if verbose:
                print(f"  neuron {i+1}: h = {h:+g}  ->  s{i+1} = {new:+d}" + ("   (changed!)" if new != s[i] else ""))
            if new != s[i]:
                s[i], changed = new, True
            energies.append(float(energy(W, s)))
        if not changed:
            break
    return s, energies

## 2. ตัวอย่างการคำนวณด้วยมือ (4 เซลล์)

### ขั้นที่ 1 — จำรูปแบบ $\mathbf{p} = (+1, -1, +1, -1)$

คำนวณ $\mathbf{p}\mathbf{p}^{\top}$ (แต่ละช่อง $= p_i \times p_j$) แล้วตั้งเส้นทแยงเป็น 0:

$$
\mathbf{p}\mathbf{p}^{\top} =
\begin{bmatrix} 1 & -1 & 1 & -1 \\ -1 & 1 & -1 & 1 \\ 1 & -1 & 1 & -1 \\ -1 & 1 & -1 & 1 \end{bmatrix}
\;\Rightarrow\;
W =
\begin{bmatrix} 0 & -1 & 1 & -1 \\ -1 & 0 & -1 & 1 \\ 1 & -1 & 0 & -1 \\ -1 & 1 & -1 & 0 \end{bmatrix}
$$

อ่านความหมาย: $w_{13} = +1$ → เซลล์ 1 กับ 3 "ควรเหมือนกัน", $w_{12} = -1$ → เซลล์ 1 กับ 2 "ควรตรงข้ามกัน"

### ขั้นที่ 2 — ป้อนข้อมูลที่เสียหาย: พลิกเซลล์ที่ 2 → $\mathbf{s} = (+1, \mathbf{+1}, +1, -1)$

**พลังงานก่อนแก้:** คำนวณ $W\mathbf{s}$ ทีละแถว
- แถว 1: $0(1) + (-1)(1) + 1(1) + (-1)(-1) = 1$
- แถว 2: $(-1)(1) + 0(1) + (-1)(1) + 1(-1) = -3$
- แถว 3: $1(1) + (-1)(1) + 0(1) + (-1)(-1) = 1$
- แถว 4: $(-1)(1) + 1(1) + (-1)(1) + 0(-1) = -1$

$\mathbf{s}^{\top}(W\mathbf{s}) = (1)(1) + (1)(-3) + (1)(1) + (-1)(-1) = 0$ → $E = -\tfrac{1}{2}(0) = \mathbf{0}$

### ขั้นที่ 3 — อัปเดตทีละเซลล์ (ลำดับ 1 → 4)

| เซลล์ | $h_i = \sum_j w_{ij} s_j$ | เครื่องหมาย | สถานะหลังอัปเดต $\mathbf{s}$ |
|:-:|---|:-:|---|
| 1 | $0 - 1 + 1 + 1 = 1$ | + | $(+1, +1, +1, -1)$ ไม่เปลี่ยน |
| 2 | $-1 + 0 - 1 - 1 = -3$ | − | $(+1, \mathbf{-1}, +1, -1)$ **เปลี่ยน!** |
| 3 | $1 + 1 + 0 + 1 = 3$ | + | ไม่เปลี่ยน |
| 4 | $-1 - 1 - 1 + 0 = -3$ | − | ไม่เปลี่ยน |

ได้ $(+1, -1, +1, -1) = \mathbf{p}$ คืนมาแล้ว! 🎉

**พลังงานหลังแก้:** $W\mathbf{p} = (3, -3, 3, -3)$ → $\mathbf{p}^{\top}W\mathbf{p} = 3+3+3+3 = 12$ → $E = -6$

พลังงานลดลงจาก $0 \to -6$ ตามทฤษฎี — รันโค้ดเพื่อตรวจ:

In [ ]:
p = np.array([1, -1, 1, -1])
W = hebbian_weights([p])
print("W =\n", W)

s = np.array([1, 1, 1, -1])
print("\nW @ s =", W @ s, "   E(s) =", energy(W, s))
print("\nAsynchronous update (sweep 1):")
s_out, E_trace = recall(W, s, verbose=True, max_sweeps=1)
print("\nผลลัพธ์:", s_out, "  เท่ากับรูปแบบที่จำไว้?", np.array_equal(s_out, p))
print("W @ p =", W @ p, "   E(p) =", energy(W, p))
print("พลังงานหลังแต่ละการอัปเดต:", E_trace)

### ข้อสังเกต: "ความจำปลอม" (spurious state)

รูปแบบกลับขั้ว $-\mathbf{p} = (-1, +1, -1, +1)$ ก็มีพลังงาน $-6$ เท่ากัน เพราะ $(-\mathbf{s})^{\top}W(-\mathbf{s}) = \mathbf{s}^{\top}W\mathbf{s}$
ถ้าข้อมูลเสียหายเกินครึ่ง เครือข่ายอาจไหลไปหา $-\mathbf{p}$ แทน

In [ ]:
print("E(p)  =", energy(W, p))
print("E(-p) =", energy(W, -p))
very_noisy = np.array([-1, 1, -1, -1])
out, _ = recall(W, very_noisy)
print(f"\nป้อน {very_noisy} (ผิดไป 3 จาก 4 ตำแหน่ง)  ->  ได้ {out}  = -p ? {np.array_equal(out, -p)}")

## 3. ตัวอย่างที่เห็นภาพ: จำตัวอักษร 10×10 = 100 เซลล์

1. จำตัวอักษร 3 ตัว (แต่ละพิกเซล: ดำ = +1, ขาว = −1)
2. ทำลายภาพโดยสุ่มพลิกพิกเซล 25%
3. ให้เครือข่ายนึกคืน

In [ ]:
LETTERS = {
    "T": ["##########",
          "##########",
          "....##....",
          "....##....",
          "....##....",
          "....##....",
          "....##....",
          "....##....",
          "....##....",
          "....##...."],
    "H": ["##......##",
          "##......##",
          "##......##",
          "##......##",
          "##########",
          "##########",
          "##......##",
          "##......##",
          "##......##",
          "##......##"],
    "Z": ["##########",
          "##########",
          ".......##.",
          "......##..",
          ".....##...",
          "....##....",
          "...##.....",
          "..##......",
          "##########",
          "##########"],
}
names = list(LETTERS)
patterns = [np.array([1 if ch == "#" else -1 for row in LETTERS[k] for ch in row]) for k in names]
W_img = hebbian_weights(patterns, normalize=True)

rng = np.random.default_rng(42)
noise = 0.25
fig, axes = plt.subplots(len(names), 3, figsize=(7, 7))
recalled_traces = []
for r, (name, pat) in enumerate(zip(names, patterns)):
    flip = rng.random(pat.size) < noise
    noisy = np.where(flip, -pat, pat)
    out, E_trace = recall(W_img, noisy, rng=rng)
    recalled_traces.append((name, E_trace))
    for c, (img, title) in enumerate([(pat, f"stored '{name}'"),
                                      (noisy, f"{flip.sum()} pixels flipped"),
                                      (out, "recalled: " + ("OK" if np.array_equal(out, pat) else f"{(out != pat).sum()} wrong"))]):
        axes[r, c].imshow(img.reshape(10, 10), cmap="gray_r", vmin=-1, vmax=1)
        axes[r, c].set_title(title, fontsize=9)
        axes[r, c].axis("off")
plt.suptitle("Hopfield network: content-addressable memory")
plt.tight_layout()
plt.show()

### พลังงานลดลงทุกครั้งที่อัปเดต

กราฟแสดงค่า $E$ หลังการอัปเดตเซลล์แต่ละครั้ง — **ไม่มีจุดไหนที่เพิ่มขึ้น** และหยุดเมื่อถึงก้นหุบเขา (รูปแบบที่จำไว้)

In [ ]:
plt.figure(figsize=(8, 4))
for name, E_trace in recalled_traces:
    plt.plot(E_trace, label=f"'{name}'")
    assert all(b <= a + 1e-9 for a, b in zip(E_trace, E_trace[1:])), "energy increased!"
plt.xlabel("single-neuron update step"); plt.ylabel("energy E")
plt.title("Energy never increases during asynchronous recall")
plt.legend()
plt.show()
print("ตรวจแล้ว: พลังงานไม่เพิ่มขึ้นเลยในทุกขั้นของทุกตัวอักษร")

## 4. ความจุ (Capacity): จำได้กี่รูปแบบ?

Hopfield network จำรูปแบบสุ่มได้ประมาณ $P_{\max} \approx 0.138\,N$ รูปแบบ (Amit, Gutfreund & Sompolinsky, 1985)
ถ้าเกินนี้ รูปแบบต่างๆ จะ "รบกวนกัน" (crosstalk) จนนึกไม่ออก

**คำนวณ:** $N = 100$ → จำได้ราว $0.138 \times 100 \approx 14$ รูปแบบ — มาทดลองดู

In [ ]:
N = 100
rng = np.random.default_rng(7)
P_values = list(range(2, 31, 2))
success = []
for P in P_values:
    ok, total = 0, 0
    for trial in range(3):
        pats = rng.choice([-1, 1], size=(P, N))
        Wc = hebbian_weights(pats, normalize=True)
        for pat in pats:
            flip = rng.random(N) < 0.05
            out, _ = recall(Wc, np.where(flip, -pat, pat), rng=rng)
            ok += np.array_equal(out, pat)
            total += 1
    success.append(ok / total)

print(pd.DataFrame({"P (patterns)": P_values, "P/N": np.array(P_values) / N,
                    "recalled perfectly": [f"{v:.0%}" for v in success]}).to_string(index=False))

plt.figure(figsize=(8, 4))
plt.plot(np.array(P_values) / N, success, "o-")
plt.axvline(0.138, color="r", ls="--", label="theory: P/N = 0.138")
plt.xlabel("load P/N"); plt.ylabel("fraction recalled perfectly")
plt.title(f"Hopfield capacity (N = {N}, 5% noise)")
plt.legend()
plt.show()

## 5. แบบฝึกหัด

1. จำ 2 รูปแบบ $\mathbf{p}^1 = (1, 1, -1, -1)$ และ $\mathbf{p}^2 = (1, -1, 1, -1)$ — คำนวณ $W$ ด้วยมือ (ใบ้: $W = \mathbf{p}^1\mathbf{p}^{1\top} + \mathbf{p}^2\mathbf{p}^{2\top}$ แล้วตั้งเส้นทแยงเป็น 0)
2. ป้อน $\mathbf{s} = (1, 1, -1, 1)$ ซึ่งต่างจาก $\mathbf{p}^1$ แค่ตำแหน่งที่ 4 — อัปเดตทีละเซลล์ลำดับ 1 → 4 ด้วยมือ ได้ $\mathbf{p}^1$ คืนมาไหม? พลังงานก่อน/หลังเท่าไร?
3. ถ้าผลไม่ใช่ $\mathbf{p}^1$ — อธิบายว่าเกิดอะไรขึ้น: เครือข่ายนี้มีสถานะเสถียรทั้งหมดกี่แบบ? (ใบ้: ความจุ $\approx 0.138 \times 4 \approx 0.55$ รูปแบบ แต่เราบังคับจำถึง 2) แล้วลองอัปเดตลำดับ 4 → 1 แทน
4. เพิ่ม noise ในตัวอย่างตัวอักษรเป็น 40% — ยังนึกออกไหม? ถ้าเกิน 50% จะเกิดอะไรขึ้น?
5. เพิ่มตัวอักษรตัวที่ 4, 5, … ด้วยตัวเอง — เครือข่ายเริ่มสับสนเมื่อไร? ทำไมตัวอักษรที่ "หน้าตาคล้ายกัน" ถึงจำยากกว่ารูปแบบสุ่ม?

รันเซลล์ถัดไปเพื่อตรวจคำตอบข้อ 1–3

In [ ]:
from itertools import product

p1, p2 = np.array([1, 1, -1, -1]), np.array([1, -1, 1, -1])
W2 = hebbian_weights([p1, p2])
print("ข้อ 1:  W =\n", W2)

s = np.array([1, 1, -1, 1])
print(f"\nข้อ 2:  ป้อน s = {s},  E(s) = {energy(W2, s):g}")
out, E_trace = recall(W2, s, verbose=True)
print(f"ได้ {out}  = p1? {np.array_equal(out, p1)}  = -p2? {np.array_equal(out, -p2)}   พลังงาน: {E_trace[0]:g} -> {E_trace[-1]:g}")

print("\nข้อ 3:  สถานะเสถียรทั้งหมด (ลองครบ 2^4 = 16 แบบ)")
names_ = [("p1", p1), ("-p1", -p1), ("p2", p2), ("-p2", -p2)]
for bits in product([-1, 1], repeat=4):
    st = np.array(bits)
    h = W2 @ st
    if np.all((h * st > 0) | (h == 0)):
        label = next((n for n, q in names_ if np.array_equal(st, q)), "spurious")
        print(f"   {st}   E = {energy(W2, st):g}   ({label},  ห่างจาก s {(st != s).sum()} ตำแหน่ง)")

s_rev = s.copy()
for i in [3, 2, 1, 0]:
    h = W2[i] @ s_rev
    s_rev[i] = s_rev[i] if h == 0 else int(np.sign(h))
print(f"\nอัปเดตลำดับ 4 -> 1 แทน: ได้ {s_rev}  = p1? {np.array_equal(s_rev, p1)}")

**เฉลยข้อ 3:** มีสถานะเสถียร 4 แบบ ($\pm\mathbf{p}^1, \pm\mathbf{p}^2$) พลังงานเท่ากันหมด และ $\mathbf{s}$ ห่างจาก $\mathbf{p}^1$ **และ** $-\mathbf{p}^2$ แค่ 1 ตำแหน่งเท่ากัน —
จึงไม่มีคำตอบที่ "ถูก" ชัดเจน ลำดับการอัปเดตเป็นตัวตัดสิน นี่คืออาการของการจำเกินความจุ: 4 เซลล์เล็กเกินกว่าจะแยก 2 ความทรงจำออกจากกันได้ชัด

## 6. สรุปเส้นทางประวัติศาสตร์

| ปี | ผู้คิดค้น | สิ่งที่เพิ่มขึ้นมา |
|---|---|---|
| 1943 | McCulloch & Pitts | เซลล์ประสาทเทียม = ประตูตรรกะ (ไม่เรียนรู้) |
| 1958 | Rosenblatt | Perceptron **เรียนรู้ weight** จากตัวอย่าง |
| 1969 | Minsky & Papert | ชี้ขีดจำกัด XOR → AI Winter |
| **1982** | **Hopfield** | เครือข่ายเป็น **หน่วยความจำ** + มุมมอง **พลังงาน** จากฟิสิกส์ |
| 1985 | Hinton & Sejnowski | **Boltzmann Machine** — Hopfield network แบบสุ่ม ที่เรียนรู้ได้ |
| 1986 | Rumelhart, Hinton & Williams | **Backpropagation** ฝึก multi-layer network ได้จริง |
| 2024 | Hopfield & Hinton | **Nobel Prize in Physics** |

> 💡 แนวคิด "หน่วยความจำที่ค้นหาด้วยเนื้อหา" ของ Hopfield ยังอยู่ในโมเดลปัจจุบัน — งานวิจัย *"Hopfield Networks is All You Need"* (2020) แสดงว่ากลไก **attention** ใน Transformer เทียบได้กับ Hopfield network รุ่นใหม่ (modern Hopfield network)